# 1st LAB: Working with Documents

You can mount a google drive to connect you drive storage or start this notebook and temporarily add the file

> **BEWARE: if you use the temporary storage the file will be removed after the notebook is closed!**




In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

import pandas as pd

In [ ]:
!ls gdrive/MyDrive/Labs/LAB1_TMS

data  L1p1_TMS_text_preprocessing.ipynb


# Importing the data

We use two datasets:

**1. SMS Spam Collection** (`spam.csv`, from your Drive): 5,572 SMS in English.

| column | meaning | example |
|---|---|---|
| `v1` | label: `ham` = normal message, `spam` = advertising or scam | `spam` |
| `v2` | text of the SMS | `Free entry in 2 a wkly comp to win FA Cup final...` |
| `Unnamed: 2/3/4` | pieces of text split by mistake (see below) | `NaN` |

After cleaning we rename `v1` → `label` and `v2` → `text`.

**2. Movie Reviews** (`nltk.corpus.movie_reviews`, Pang & Lee 2004): 2,000 movie reviews, 1,000 positive (`pos`) and 1,000 negative (`neg`).

- `movie_reviews.raw()` returns **all** the reviews joined in a single string, without the pos/neg label.
- The text is already lowercase and the punctuation is already separated from the words (`party , drink`), so some of the steps below change little on it.

In [ ]:
## Reading the SMS Spam Collection from a csv file

# base_path = ''
base_path = 'gdrive/MyDrive/Labs/LAB1_TMS/data/' # use the path to your gdrive folder

data_spam = pd.read_csv(f"{base_path}spam.csv", encoding="ISO-8859-1")
display(data_spam.head())
print(data_spam.shape)

In [ ]:
display(data_spam[data_spam['Unnamed: 2'].notna()].head())
print(data_spam[data_spam['Unnamed: 2'].notna()].shape)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
95,spam,Your free ringtone is waiting to be collected....,PO Box 5249,"MK17 92H. 450Ppw 16""",NaN
281,ham,\Wen u miss someone,the person is definitely special for u..... B...,why to miss them,"just Keep-in-touch\"" gdeve.."""
444,ham,\HEY HEY WERETHE MONKEESPEOPLE SAY WE MONKEYAR...,HOWU DOIN? FOUNDURSELF A JOBYET SAUSAGE?LOVE ...,NaN,NaN
671,spam,SMS. ac sun0819 posts HELLO:\You seem cool,"wanted to say hi. HI!!!\"" Stop? Send STOP to ...",NaN,NaN
710,ham,Height of Confidence: All the Aeronautics prof...,"this wont even start........ Datz confidence..""",NaN,NaN


(50, 5)


The columns `Unnamed: 2`, `Unnamed: 3` and `Unnamed: 4` are almost always empty.
In 50 rows, however, the text of the SMS contains commas and was split across them by the csv reader.

For example, row 95 is: `Your free ringtone is waiting to be collected....` + `PO Box 5249` + `MK17 92H. 450Ppw 16"`

If we kept only `v2` we would lose the end of these messages, so we join the pieces back together.

In [ ]:
# Join each message back into a single text: the csv was split on the commas
text_cols = ['v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
data_spam['v2'] = data_spam[text_cols].apply(lambda row: ','.join(row.dropna()), axis=1)

# Keep only the label and the text, with readable names
data_spam = data_spam[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})
display(data_spam.loc[[95, 281]])
print(data_spam.shape)

In [ ]:
import nltk
from nltk.corpus import movie_reviews
nltk.download('movie_reviews') ###< run once
nltk.download('punkt') ###< run once
nltk.download('punkt_tab') ###< run once, needed by recent NLTK versions

Data for google colab gets saved here:

`/root/nltk_data/corpora/movie_reviews`

In [ ]:
!ls /root/nltk_data/corpora/movie_reviews

neg  pos  README


What is this dataset all about?

In [ ]:
print(movie_reviews.readme()[:500])
print('...')

Sentiment Polarity Dataset Version 2.0
Bo Pang and Lillian Lee

http://www.cs.cornell.edu/people/pabo/movie-review-data/

Distributed with NLTK with permission from the authors.


Introduction

This README v2.0 (June, 2004) for the v2.0 polarity dataset comes from
the URL http://www.cs.cornell.edu/people/pabo/movie-review-data .


What's New -- June, 2004

This dataset represents an enhancement of the review corpus v1.0
described in README v1.1: it contains more reviews, and labe
...


In [ ]:
raw = movie_reviews.raw()
print(f"RAW representation of movies reviews dataset, number of words {len(movie_reviews.words())}\n")
print(raw[:500])

# Remove whitespaces

In [ ]:
print('Sample Text:\n')
text = "DEAR FRIEND,\n\nI AM ( DR.) JAMES NGOLA, THE PERSONAL ASSISTANCE TO THE LATE CONGOLESE (PRESIDENT LAURENT KABILA) WHO WAS ASSASSINATED BY HIS BODY GUARD ON 16TH JAN. 2001.\n\n\nTHE INCIDENT OCCURRED IN OUR PRESENCE WHILE WE WERE HOLDING MEETING WITH HIS EXCELLENCY OVER THE FINANCIAL RETURNS FROM THE DIAMOND SALES IN THE AREAS CONTROLLED BY (D.R.C.) DEMOCRATIC REPUBLIC OF CONGO FORCES AND THEIR FOREIGN ALLIES ANGOLA AND ZIMBABWE, HAVING RECEIVED THE PREVIOUS DAY (USD$100M) ONE HUNDRED MILLION UNITED STATES DOLLARS, CASH IN THREE DIPLOMATIC BOXES ROUTED THROUGH ZIMBABWE.\n\nMY PURPOSE OF WRITING YOU THIS LETTER IS TO SOLICIT FOR YOUR ASSISTANCE AS TO BE A COVER TO THE FUND AND ALSO COLLABORATION IN MOVING THE SAID FUND INTO YOUR BANK ACCOUNT THE SUM OF (USD$25M) TWENTY FIVE MILLION UNITED STATES DOLLARS ONLY, WHICH I DEPOSITED WITH A SECURITY COMPANY IN GHANA, IN A DIPLOMATIC BOX AS GOLDS WORTH (USD$25M) TWENTY FIVE MILLION UNITED STATES DOLLARS ONLY FOR SAFE KEEPING IN A SECURITY VAULT FOR ANY FURTHER INVESTMENT PERHAPS IN YOUR COUNTRY. \n\nYOU WERE INTRODUCED TO ME BY A RELIABLE FRIEND OF MINE WHO IS A TRAVELLER,AND ALSO A MEMBER OF CHAMBER OF COMMERCE AS A RELIABLE AND TRUSTWORTHY PERSON WHOM I CAN RELY ON AS FOREIGN PARTNER, EVEN THOUGH THE NATURE OF THE TRANSACTION WAS NOT REVEALED TO HIM FOR SECURITY REASONS.\n\n\nTHE (USD$25M) WAS PART OF A PROCEEDS FROM DIAMOND TRADE MEANT FOR THE LATE PRESIDENT LAURENT KABILA WHICH WAS DELIVERED THROUGH ZIMBABWE IN DIPLOMATIC BOXES. THE BOXES WERE KEPT UNDER MY CUSTODY BEFORE THE SAD EVENT THAT TOOK THE LIFE OF (MR. PRESIDENT).THE CONFUSION THAT ENSUED AFTER THE ASSASSINATION AND THE SPORADIC SHOOTING AMONG THE FACTIONS, I HAVE TO RUN AWAY FROM THE COUNTRY FOR MY DEAR LIFE AS I AM NOT A SOLDIER BUT A CIVIL SERVANT I CROSSED RIVER CONGO TO OTHER SIDE OF CONGO LIBREVILLE FROM THERE I MOVED TO THE THIRD COUNTRY GHANA WHERE I AM PRESENTLY TAKING REFUGE. \n\nAS A MATTER OF FACT, WHAT I URGENTLY NEEDED FROM YOU IS YOUR ASSISTANCE IN MOVING THIS MONEY INTO YOUR ACCOUNT IN YOUR COUNTRY FOR INVESTMENT WITHOUT RAISING EYEBROW. FOR YOUR ASSISTANCE I WILL GIVE YOU 20% OF THE TOTAL SUM AS YOUR OWN SHARE WHEN THE MONEY GETS TO YOUR ACCOUNT, WHILE 75% WILL BE FOR ME, OF WHICH WITH YOUR KIND ADVICE I HOPE TO INVEST IN PROFITABLE VENTURE IN YOUR COUNTRY IN OTHER TO SETTLE DOWN FOR MEANINGFUL LIFE, AS I AM TIRED OF LIVING IN A WAR ENVIRONMENT. \n\nTHE REMAINING 5% WILL BE USED TO OFFSET ANY COST INCURRED IN THE CAUSE OF MOVING THE MONEY TO YOUR ACCOUNT. IF THE PROPOSAL IS ACCEPTABLE TO YOU PLEASE CONTACT ME IMMEDIATELY THROUGH THE ABOVE TELEPHONE AND E-MAIL, TO ENABLE ME ARRANGE FACE TO FACE MEETING WITH YOU IN GHANA FOR THE CLEARANCE OF THE FUNDS BEFORE TRANSFRING IT TO YOUR BANK ACCOUNT AS SEEING IS BELIEVING. \n\nFINALLY, IT IS IMPORTANT ALSO THAT I LET YOU UNDERSTAND THAT THERE IS NO RISK INVOLVED WHATSOEVER AS THE MONEY HAD NO RECORD IN KINSHASA FOR IT WAS MEANT FOR THE PERSONAL USE OF (MR. PRESIDEND ) BEFORE THE NEFARIOUS INCIDENT OCCURRED, AND ALSO I HAVE ALL THE NECESSARY DOCUMENTS AS REGARDS TO THE FUNDS INCLUDING THE (CERTIFICATE OF DEPOSIT), AS I AM THE DEPOSITOR OF THE CONSIGNMENT.\n\n\nLOOKING FORWARD TO YOUR URGENT RESPONSE.\n\nYOUR SINCERELY,\n\n\t\t\tMR. JAMES NGOLA. "
print(text)

Sample Text:

DEAR FRIEND,

I AM ( DR.) JAMES NGOLA, THE PERSONAL ASSISTANCE TO THE LATE CONGOLESE (PRESIDENT LAURENT KABILA) WHO WAS ASSASSINATED BY HIS BODY GUARD ON 16TH JAN. 2001.


THE INCIDENT OCCURRED IN OUR PRESENCE WHILE WE WERE HOLDING MEETING WITH HIS EXCELLENCY OVER THE FINANCIAL RETURNS FROM THE DIAMOND SALES IN THE AREAS CONTROLLED BY (D.R.C.) DEMOCRATIC REPUBLIC OF CONGO FORCES AND THEIR FOREIGN ALLIES ANGOLA AND ZIMBABWE, HAVING RECEIVED THE PREVIOUS DAY (USD$100M) ONE HUNDRED MILLION UNITED STATES DOLLARS, CASH IN THREE DIPLOMATIC BOXES ROUTED THROUGH ZIMBABWE.

MY PURPOSE OF WRITING YOU THIS LETTER IS TO SOLICIT FOR YOUR ASSISTANCE AS TO BE A COVER TO THE FUND AND ALSO COLLABORATION IN MOVING THE SAID FUND INTO YOUR BANK ACCOUNT THE SUM OF (USD$25M) TWENTY FIVE MILLION UNITED STATES DOLLARS ONLY, WHICH I DEPOSITED WITH A SECURITY COMPANY IN GHANA, IN A DIPLOMATIC BOX AS GOLDS WORTH (USD$25M) TWENTY FIVE MILLION UNITED STATES DOLLARS ONLY FOR SAFE KEEPING IN A SECURITY

In [ ]:
print('Initial text lenght:', len(text))
print("Number of 2 newlines", text.count('\n\n'))
print("Number of tabs", text.count('\t'))
print("Number of 2 tabs", text.count('\t\t'))
print("Number of tab and newline", text.count('\n\t'))

Initial text lenght: 3237
Number of 2 newlines 11
Number of tabs 3
Number of 2 tabs 1
Number of tab and newline 1


In [ ]:
print('Manual approach:')
# replace every newline and every tab with a space
new_text = text.replace('\n', ' ').replace('\t', ' ')
print(new_text)
print("Number of newlines", new_text.count('\n'))
print("Number of tabs", new_text.count('\t'))
print("Number of 2 spaces", new_text.count('  '))
print('Manual text lenght', len(new_text))

### What does the manual approach do, and what does it miss?

Each `\n` and each `\t` becomes **one** space, so the words are never glued. But a sequence of whitespaces becomes a sequence of spaces:

| original text | after `.replace('\n', ' ').replace('\t', ' ')` |
|---|---|
| `FRIEND,\n\nI AM` | `FRIEND,  I AM` (2 spaces) |
| `2001.\n\n\nTHE` | `2001.   THE` (3 spaces) |
| `SINCERELY,\n\n\t\t\tMR.` | `SINCERELY,     MR.` (5 spaces) |

Adding `.replace('\n\n', ' ')` at the end does not help: at that point there are no `\n` left, they have already been replaced.
`replace` only works with a fixed string, so it cannot say "any number of whitespaces". A regex can: let's try it.

In [ ]:
print('\nRegex library (WRONG version, see below):')
import re
# \s\s matches 2 whitespaces at a time and replaces them with nothing
new_text2 = re.sub(r'\s\s', '', text)
print(new_text2)
print("Number of 2 newlines", new_text2.count('\n\n'))
print("Number of tabs", new_text2.count('\t'))
print("Number of 2 tabs", new_text2.count('\t\t'))
print("Number of tab and newline", new_text2.count('\n\t'))
print('Regex text lenght', len(new_text2))

### Why is `re.sub(r'\s\s', '', text)` wrong?

`\s\s` matches **exactly 2** whitespaces (spaces, `\n`, `\t`) and replaces them with **nothing** (`''`). This causes 3 problems, all visible in the output above:

| problem | original text | after `\s\s` → `''` |
|---|---|---|
| 2 whitespaces are removed → the words get glued | `FRIEND,\n\nI AM` | `FRIEND,I AM` |
| 3 whitespaces: 2 are removed, 1 is left | `2001.\n\n\nTHE` | `2001.\nTHE` |
| the leftover is still a `\t` or `\n`, not a space | `SINCERELY,\n\n\t\t\tMR.` | `SINCERELY,\tMR.` |

The glued words are a problem later: when we split the text into words, `FRIEND,I` becomes a single token.

The fix (as in the slides):
- `\s+` matches **1 or more** whitespaces, however many there are;
- we replace them with **one space** `' '`, not with nothing.

In [ ]:
# Correct version 1: regex
# \s+ = one or more whitespaces, replaced by a single space
clean_text = re.sub(r'\s+', ' ', text)

# Correct version 2: string methods
# split() cuts on any whitespace, " ".join() puts the words back with one space
# (it also removes the whitespaces at the beginning and at the end)
clean_text2 = " ".join(text.split())

print(clean_text)
print("Number of 2 newlines", clean_text.count('\n\n'))
print("Number of tabs", clean_text.count('\t'))
print('Regex text lenght', len(clean_text))
print('String methods text lenght', len(clean_text2))

text = clean_text

##Exercise
How many '\n' can be found in the first 2000 chars of the movie dataset?

In [ ]:
# EXERCISE: do something here

movie_text = movie_reviews.raw()[:2000]

print("Number of 1 newline", movie_text.count('\n'))
print("Number of 2 newlines", movie_text.count('\n\n'))
print("Number of tabs", movie_text.count('\t'))
print("Number of 2 tabs", movie_text.count('\t\t'))
print("Number of tab and newline", movie_text.count('\n\t'))
print('Movie text lenght', len(movie_text))

# Case folding



In [ ]:
print("Initial text:")
print(text[:110])
lower = text.lower()

print()
print("Final text:")
print(lower[:110])

Initial text:
plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the

Final text:
plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the


# Remove punctuation and other Symbols

Every method to remove punctuation answers two questions:

1. **Which characters** are removed?
2. **What do we put in their place**: nothing `''` or a space `' '`?

Different answers give different results. We try two methods on the same examples and compare them.

In [ ]:
# Method 1: string.punctuation, a FIXED list of 32 ASCII symbols
import string
print("Library's punctuation: ", string.punctuation)

# The list does not contain '€', so we build a custom one
custom = list(string.punctuation)

# EXERCISE: do something here, add the '€' symbol to the custom list
custom.append('€')

print('Custom list:', custom,'\n')

# Defining the functions to remove punctuation: keep only the characters NOT in the list
def remove_punctuation(text):
    punctuationfree = "".join([i for i in text if i not in string.punctuation])
    return punctuationfree

def remove_custom_punctuation(text):
    punctuationfree = "".join([i for i in text if i not in custom])
    return punctuationfree

sm_text= 'Everything is ok! €€'
print("Simple Example:          ", sm_text)
print("With string.punctuation: ", remove_punctuation(sm_text))
print("With the custom list:    ", remove_custom_punctuation(sm_text))

In [ ]:
# Method 2: regex (as in the slides)
# [^\w\s] = any character that is NOT a letter/digit/underscore (\w) and NOT a whitespace (\s)
# We do not list the symbols to remove: we list the characters to KEEP
def remove_punctuation_regex(text, replace_with=''):
    return re.sub(r'[^\w\s]', replace_with, text)

print("Simple Example:", sm_text)
print("With the regex:", remove_punctuation_regex(sm_text))

### Question 1: which characters are removed?

The two methods describe the punctuation in opposite ways:

| | Method 1: `string.punctuation` | Method 2: regex `[^\w\s]` |
|---|---|---|
| how it works | removes the characters **in a list** | removes everything that is **not** a letter, a digit, `_` or a whitespace |
| `€`, `£`, `°`, `’` (curly apostrophe) | **kept**: they are not in the list, we must add them by hand | **removed** automatically |
| `_` (underscore) | **removed**: it is in the list | **kept**: for the regex `_` is a "word" character (`\w`) |

The next cell shows these differences on real examples: `traveller,and`, `(d.r.c.)` and `(usd$100m)` come from the email above, `5k’s` from the tweet in the emoji section.

In [ ]:
examples = [
    'Everything is ok! €€',
    'traveller,and also',   # from the email
    '(d.r.c.)',             # from the email
    '(usd$100m)',           # from the email
    "don't",
    '5k’s',                 # from the tweet (curly apostrophe)
    'snake_case',
    '30°C £5',
]

# quotes around each result, so the spaces are visible
show = lambda s: f"'{s}'"

comparison = pd.DataFrame({
    'original': [show(t) for t in examples],
    'string.punctuation → ""': [show(remove_punctuation(t)) for t in examples],
    'custom list → ""': [show(remove_custom_punctuation(t)) for t in examples],
    'regex → ""': [show(remove_punctuation_regex(t)) for t in examples],
    'regex → " "': [show(remove_punctuation_regex(t, ' ')) for t in examples],
})
display(comparison)

### Question 2: what do we put in their place?

Compare the last two columns of the table (same regex, different replacement):

| original | → `''` (nothing) | → `' '` (a space) |
|---|---|---|
| `traveller,and also` | `travellerand also` ❌ two words glued | `traveller and also` ✅ |
| `(d.r.c.)` | `drc` ✅ one token | `' d r c  '` ❌ three tokens + extra spaces |
| `don't` | `dont` ✅ | `don t` ❌ two tokens |
| `5k’s` | `5ks` | `5k s` |

- `''` keeps together what belongs together (`d.r.c.`, `don't`), but **glues** words separated only by a symbol (`traveller,and`).
- `' '` keeps the words apart, but **splits** abbreviations and contractions, and leaves **extra spaces** (we must remove the whitespaces again).

**No method is right in every case**: we choose by looking at our data and at the downstream task.
In the SMS dataset many words are written with apostrophes (`don't`, `I'm`, `it's`): with `''` they stay one token, so below we use `''`.

> **Note on the slides:** the slides use `re.sub(r"[^\w\s]|\d", "", text)`. The `|\d` part also removes the **digits**: `(usd$100m)` → `usdm`, `30°C £5` → `C `.
> Here we keep the two steps apart: the digits are removed in the next section.

In [ ]:
print("Spam Example")

# Storing the punctuation free text (method 1, replacing with nothing)
data_spam['text'] = data_spam['text'].apply(lambda x: remove_punctuation(x))
display(data_spam.head(2))

# Remove numbers


In [ ]:
n_text = "In July 1776, during the American Revolutionary War, the 13 Colonies, acting jointly through the 2nd Continental.."
print(n_text)
no_num = re.sub(r'\d+', '', n_text)
print(no_num + '\n')

data_spam['text']= data_spam['text'].apply(lambda x: re.sub(r'\d+', '', x))
display(data_spam.head(5))

# Remove repeated characters


In [ ]:
def character_repetition(text):
    # Pattern matching for all alphabetic characters where any single character
    # repeats three or more times consecutively.
    # ([A-Za-z])  - Capturing group. Captures any alphabetic character.
    # \1{2,}      - Matches the captured character repeating at least two more
    # times after the initial match.
    # For example, this pattern matches 'aaa', 'BBBB', etc.
    pattern_alpha = re.compile(r"([A-Za-z])\1{2,}", re.DOTALL)

    # Limiting all the repetition to two characters.
    formatted_text = pattern_alpha.sub(r"\1\1", text)

    # Pattern matching for all the punctuations that can occur
    # EXERCISE: do something in the following line where the x is
    pattern_punct = re.compile(r'([!$?])\1{1,}')

    # Limiting punctuations in previously formatted string to only one.
    # EXERCISE: do something in the following line where the x is
    combined_formatted = pattern_punct.sub(r"\1", formatted_text)
    return combined_formatted

repeated = "Realllllllllyyyyy good!!!!!"
print("Before:", repeated)
print("After: ", character_repetition(repeated))

Before: Realllllllllyyyyy good!!!!!
After:  Reallyy good!


# Spelling Correction

In [ ]:
# !pip install textblob
from textblob import TextBlob
wrong = "raeyly good!!!!!"

def correct_me(text):
  text_blb = TextBlob(text)
  text_corrected = text_blb.correct()   # Correcting the text
  return text_corrected

print("Before:", wrong)
print("After: ", correct_me(wrong))

Before: raeyly good!!!!!
After:  really good!!!!!


In [ ]:
# Example from the slides:
wrong_slides = "repeaaaaaatedddd !!!"

print("Before:", wrong_slides)
print("After: ", correct_me(wrong_slides))

Before: repeaaaaaatedddd !!!
After:  repeaaaaaatedddd !!!


It did not correct! We need to combine it with the regex to remove multiple repetitions!

In [ ]:
print("Before:", wrong_slides)
print("After repetition removal:", character_repetition(wrong_slides))
print("After repetition removal + correction:", correct_me(character_repetition(wrong_slides)))

Before: repeaaaaaatedddd !!!
After repetition removal: repeaatedd !
After repetition removal + correction: repeated !


# Remove Links

In [ ]:
def remove_links(text):
  # Removing all the occurrences of links that starts with https
  text = re.sub(r'http\S+', ' ', text)

  # Remove all the occurrences of text that ends with .com
  text = re.sub(r"www\.[A-Za-z]*\.com", " ", text)
  return text

link_text = "For more information visit www.example.com or https://www.example.com."
print("Before:", link_text)
print("After: ", remove_links(link_text))

Before: For more information visit www.example.com or https://www.example.com.
After:  For more information visit   or  


Remember during the slides presentation we talked how there can be a lot of issues with the presented regexes, well here is an example where it fails:

In [ ]:
link_text = "Download mongoDB from mongodb.com and access it from 127.0.0.0:27020 192.168.1.1"
print("Before:", link_text)
print("After: ", remove_links(link_text))

Before: Download mongoDB from mongodb.com and access it from 127.0.0.0:27020 192.168.1.1
After:  Download mongoDB from mongodb.com and access it from 127.0.0.0:27020 192.168.1.1


## Exercise:
Think about a regex to solve the mentioned problems.

In [ ]:
link_text = "Download mongoDB from mongodb.com and access it from 127.0.0.0:27020 192.168.1.255"

def remove_links_improved(text):
  # EXERCISE: do something here
  # Removing all the occurrences of links that starts with https
  text = re.sub(r'http\S+', ' ', text)

  # Remove all the occurrences of text that ends with .com
  text = re.sub(r"www\.[A-Za-z]*\.com", " ", text)

  # Remove all the occurrences of text that ends with .com
  text = re.sub(r"[A-Za-z]*\.com", " ", text)
  # raise NotImplementedError('Implement the function before')
  return text

print("Before:", link_text)
print("After: ", remove_links_improved(link_text))

Before: Download mongoDB from mongodb.com and access it from 127.0.0.0:27020 192.168.1.255
After:  Download mongoDB from   and access it from 127.0.0.0:27020 192.168.1.255


# Removing Emoji

In [ ]:
!pip install emoji
!pip install demoji

In [ ]:
import emoji
import demoji

print("\U0001f605 is  \\U0001f605 in utf")

print('\U0001f605 is :grinning_face_with_sweat: in emoji library')
print(emoji.emojize('emojized :grinning_face_with_sweat:'))

😅 is  \U0001f605 in utf
😅 is :grinning_face_with_sweat: in emoji library
emojized 😅


In [ ]:
print("\n Working with Tweets")
tweet = """\
... #startspreadingthenews yankees win great start by 🎅🏾 going 5strong innings with 5k’s🔥 🐂
... solo homerun 🌋 🇲🇽 and 🇳🇮 to close the game🔥🔥!!!....
... WHAT A GAME!!"""

print(tweet)
display(demoji.findall(tweet))


 Working with Tweets
... #startspreadingthenews yankees win great start by 🎅🏾 going 5strong innings with 5k’s🔥 🐂
... solo homerun 🌋 🇲🇽 and 🇳🇮 to close the game🔥🔥!!!....
... WHAT A GAME!!


{'🔥': 'fire',
 '🇲🇽': 'flag: Mexico',
 '🇳🇮': 'flag: Nicaragua',
 '🐂': 'ox',
 '🎅🏾': 'Santa Claus: medium-dark skin tone',
 '🌋': 'volcano'}

# Remove HTML Tags


In [ ]:
from bs4 import BeautifulSoup

sample = 'This is a nice place to live. <IMG>Image description</IMG>'

def strip_html_tags(text):
    # Initiating BeautifulSoup object soup.
    soup = BeautifulSoup(text, "html.parser")

    # Get all the text other than html tags.
    stripped_text = soup.get_text(separator=" ")
    return stripped_text

print("Before:", sample)
print('After: ', strip_html_tags(sample))

Before: This is a nice place to live. <IMG>Image description</IMG>
After:  This is a nice place to live.  Image description


# TIME for Exercise 1.1



## 1.1

Use the data stored in the varaible **raw**.

1.   Remove whitespaces, numbers, punctuation and lower the raw_text. Save as new_text.

There is a text sample in the hidden cell named evaluation_text:

>A.   How many times does the number 1 appears in the evaluation_text? \\
>B.   Can you find the € symbol in the evaluation_text? \\
>C.   If B == True, then remove it. \\
>D.   How many times does the number 1 appears in the raw_text? \\
>E.   Remove numbers from raw_text.
```
You can use:
import string
import regex as re
text.replace('', '')
text.count(' ')
"".join([character for character in text if character not in string.punctuation])+
```

In [ ]:
evaluation_text = """it's everything a movie can be : funny , dramatic , interesting , weird , funny , weird and strikingly original .
yep that pretty much describes this movie .
it starts out like a regular movie and ends up being one of the weirdest , funniest most original movies i have ever seen .
it boggles the mind and some have to wonder why we cannot get €€ movies like this more often .
besides being one of the best films of the year , being john malkovich may as well be one of the best movies ever .
period .
then again there are so many good movies , that one cannot pick an all time favorite .
john cusack plays a puppeteer craig schwartz a man out of 1 a job , in search of a job .
his wife lotte schwartz who is played by a completely un-noticeable cameron diaz who looks like something off the streets is an animal lover and has about every kind of animal you could think of .
craig finds a job as a file at the 7 ? floor of a business€ building . . . you
have to pry 1 open the elevator doors open before it reaches floor eight , the 7 ? floor is just a floor between 7 and 8 .
he is hired by his 105 year old boss ( orsen bean ) to be a filer .
in his office , he discovers a little door €, to which was boarded up and hidden .
to his curiosity he opens it and starts to 11 crawl toward it , he then gets sucked to the end and ends up in john malkovich's mind .
fifteen minutes later he is shot out onto the side of the new jersey turnpike .
he returns to tell his co-worker maxine ( catherine keener ) that he has found a portal that will lead him into john malkovich's mind , she doesn't believe him but after she sees it it changes her mind .
lotte also finds out about the portal and discovers that being someone else is good after all .
john malkovich of course has no idea what is going on , and by the end of this bizarre film there are so many twists and turns , that we don't know what really happened .
john cusack is outstanding and utterly believeable in a role only he himself could play .
he fits the role perfectly and to me was brilliantly cast .
cameron diaz is outstanding and utterly one of the world's most prettiest women , is made up here unnoticeable and very unattractive .
she however gives a very comic performance and this could easily be her best role to date .
catherine keener is very funny and sexy as maxine and of course the best thing of the movie is the magic himself john malkovich who is very brilliant and this movie plays big time homage ? to the master himself .
the whole group combined gives us a wonderfully funny movie that is also smart and clever . """

# add your code here

# Tokenization

Tokenization is the process of breaking down a piece of text (such as a sentence or paragraph) into smaller units called tokens. These tokens can be words, phrases, individual characters, or word pieces known as sub-words, depending on the purpose and type of tokenization.

In [ ]:
import re

# Defining text to tokenize
tokenize = text[:111]
print('Before:            ', re.sub(r'\s\s', '', tokenize))

print('String Operator:   ', tokenize.split())

def tokenization(text):
    tokens = re.split('\W+',text)
    return tokens

print("Simple Regex:      ", tokenization(tokenize))


### WordPunctTokenizer
from nltk.tokenize import WordPunctTokenizer

new_text4 = WordPunctTokenizer().tokenize(tokenize)
print("WordPunctTokenizer:", new_text4)

Before:             plot : two teen couples go to a church party , drink and then drive .they get into an accident .one of the 
String Operator:    ['plot', ':', 'two', 'teen', 'couples', 'go', 'to', 'a', 'church', 'party', ',', 'drink', 'and', 'then', 'drive', '.', 'they', 'get', 'into', 'an', 'accident', '.', 'one', 'of', 'the']
Simple Regex:       ['plot', 'two', 'teen', 'couples', 'go', 'to', 'a', 'church', 'party', 'drink', 'and', 'then', 'drive', 'they', 'get', 'into', 'an', 'accident', 'one', 'of', 'the', '']
WordPunctTokenizer: ['plot', ':', 'two', 'teen', 'couples', 'go', 'to', 'a', 'church', 'party', ',', 'drink', 'and', 'then', 'drive', '.', 'they', 'get', 'into', 'an', 'accident', '.', 'one', 'of', 'the']


<>:10: SyntaxWarning: invalid escape sequence '\W'
<>:10: SyntaxWarning: invalid escape sequence '\W'
/tmp/ipykernel_3708/1847101945.py:10: SyntaxWarning: invalid escape sequence '\W'
  tokens = re.split('\W+',text)


In [ ]:
print('\nWorking with Tweets\n')
### TweetTokenizer
from nltk.tokenize import TweetTokenizer

tknzr = TweetTokenizer()
tweet_tok = "This is a cooool #dummysmiley: :-) :-P <3 and some arrows < > -> <--"
print('TweetTokenizer:', tknzr.tokenize(tweet_tok))
print("Simple Regex:  ", tokenization(tweet_tok))


Working with Tweets

TweetTokenizer: ['This', 'is', 'a', 'cooool', '#dummysmiley', ':', ':-)', ':-P', '<3', 'and', 'some', 'arrows', '<', '>', '->', '<--']
Simple Regex:   ['This', 'is', 'a', 'cooool', 'dummysmiley', 'P', '3', 'and', 'some', 'arrows', '']


# Sentence split

Sentence tokenization is useful in natural language processing (NLP) for a variety of tasks where understanding, analyzing, or processing text at the sentence level is important. Here are some reasons and applications where sentence tokenization is essential.
For example, when analyzing sentiment, some models assess the sentiment of each sentence individually before combining results for the entire text.


In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize
new_text3 = sent_tokenize(text)
display(new_text3)

['plot : two teen couples go to a church party , drink and then drive .',
 'they get into an accident .',
 'one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares .',
 "what's the deal ?",
 'watch the movie and " sorta " find out .',
 '.',
 '.',
 'critique : a mind-fuck movie for the teen generation that touches on a very cool idea , but presents it in a very bad package .',
 "which is what makes this review an even harder one to write , since i generally applaud films which attempt to break the mold , mess with your head and such ( lost highway & memento ) , but there are good and bad ways of making all types of films , and these folks just didn't snag this one correctly .",
 'they seem to have taken this pretty neat concept , but executed it terribly .',
 'so what are the problems with the movie ?',
 "well , its main problem is that it's simply too jumbled .",
 'it starts off " normal " but then downshifts into this " fantasy " world in which 

# Remove stop words

Removing stop words is a common preprocessing step in natural language processing (NLP) and text analysis. Stop words are commonly used words in a language (such as "the," "is," "in," "and") that typically carry little semantic weight and do not add much to the meaning of a sentence.
Removing them reduces computational costs, and improves model performance by eliminating words that add little value to analysis.

However, there are cases where it may not be ideal.
For very short texts, like tweets, stop words might carry critical meaning.
In question-answering systems or sentiment analysis, stop words like “not” or “but” can influence sentiment and should not be removed.

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from nltk.tokenize import WordPunctTokenizer

stop_words = nltk.corpus.stopwords.words('english')

def remove_stopwords(text):
  tokenized_text = WordPunctTokenizer().tokenize(text)

  # EXERCISE: Return the sentence without stopwords
  tokenized_text_without_stopwords = [w for w in tokenized_text if not w.lower() in stop_words]
  return tokenized_text_without_stopwords

stop_me = "We are ready to travel to NY."
print('With Stopwords:   ', (stop_me.split()))
print('Without Stopwords:', remove_stopwords(stop_me))

# Remove stopwords from the movie reviews
print('Movie Reviews: Total tokens with stopwords:   ', len(WordPunctTokenizer().tokenize(raw)))
clean_re = remove_stopwords(raw)
print('Movie Reviews: Total tokens without stopwords:', len(clean_re))

With Stopwords:    ['We', 'are', 'ready', 'to', 'travel', 'to', 'NY.']
Without Stopwords: ['ready', 'travel', 'NY', '.']
Movie Reviews: Total tokens with stopwords:    1583820
Movie Reviews: Total tokens without stopwords: 955610


# Stemming

Stemming is a text preprocessing technique used in natural language processing (NLP) to reduce words to their root or base form. This root form is called the stem. Stemming algorithms typically remove common word endings, such as -ing, -ed, -s, and -ly, to bring words with similar meanings down to a shared base form.

In [ ]:
# Importing the Stemming function from nltk library
from nltk.stem.porter import PorterStemmer

# Defining the object for stemming
porter_stemmer = PorterStemmer()

# Defining a function for stemming
def stemming(text):
  stem_text = [porter_stemmer.stem(word) for word in text]
  return stem_text

data_spam['no_stopwords'] = data_spam['text'].apply(lambda x: remove_stopwords(x))

# EXERCISE: Stem the column no_stopwords:
data_spam['msg_stemmed'] = data_spam['no_stopwords'].apply(lambda x: stemming(x))
display(data_spam.head(5))

# Lemmatization

Lemmatization is a more sophisticated process that reduces words to their lemma, which is the dictionary or canonical form of a word. It involves considering the context and part of speech (POS) of each word to ensure accurate transformation.

In [ ]:
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
from nltk.stem import WordNetLemmatizer

# Defining the object for Lemmatization
wordnet_lemmatizer = WordNetLemmatizer()

# Defining the function for lemmatization
def lemmatizer(text):
  # EXERCISE: Lemmatize each word with wordnet_lemmatizer.lemmatize(
  # replacing the x
  lemm_text = [wordnet_lemmatizer.lemmatize(word) for word in text]
  return lemm_text

# EXERCISE: Lemmatize the column no_stopwords:
data_spam['msg_lemmatized'] = data_spam['no_stopwords'].apply(lambda x: lemmatizer(x))
display(data_spam.head())

,v1,v2,no_stopwords,msg_stemmed,msg_lemmatized
0,ham,Go until jurong point crazy Available only in ...,"[Go, jurong, point, crazy, Available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre...","[Go, jurong, point, crazy, Available, bugis, n..."
1,ham,Ok lar Joking wif u oni,"[Ok, lar, Joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]","[Ok, lar, Joking, wif, u, oni]"
2,spam,Free entry in a wkly comp to win FA Cup final...,"[Free, entry, wkly, comp, win, FA, Cup, final,...","[free, entri, wkli, comp, win, fa, cup, final,...","[Free, entry, wkly, comp, win, FA, Cup, final,..."
3,ham,U dun say so early hor U c already then say,"[U, dun, say, early, hor, U, c, already, say]","[u, dun, say, earli, hor, u, c, alreadi, say]","[U, dun, say, early, hor, U, c, already, say]"
4,ham,Nah I dont think he goes to usf he lives aroun...,"[Nah, dont, think, goes, usf, lives, around, t...","[nah, dont, think, goe, usf, live, around, tho...","[Nah, dont, think, go, usf, life, around, though]"


# TIME for another Exercise

## 1.2  
Use any tokenization method, you prefer to tokenize the **raw** and the **new_text**.
>A.   How many tokens do they have? \\
>B.   How many sentences do they have? \\  
>C.   Apply **Stemming** and **Lemmatization** and then count the **tokens**.

```
# You can use:
from nltk.tokenize import WordPunctTokenizer
WordPunctTokenizer().tokenize(tokenize)
from nltk.tokenize import TweetTokenizer
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
from nltk.stem.porter import PorterStemmer
#defining the object for stemming
porter_stemmer = PorterStemmer()

#For unique members of a list use set(list_name)
```




In [ ]:
# add your code here


# Part-of-speech Tagging

Part-of-Speech (POS) Tagging is a process in natural language processing (NLP) that involves assigning grammatical tags, or parts of speech, to each word in a sentence. These tags represent the word's syntactic role, such as noun, verb, adjective, adverb, etc.

In [ ]:
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

from textblob import TextBlob

tag = [
    "Sukanya, Rajib and Naba are my good friends. " ,
    "Sukanya is getting married next year. " ,
    "Marriage is a big step in one’s life." ,
    "It is both exciting and frightening. " ,
    "But friendship is a sacred bond between people." ,
    "It is a special kind of love between us. ",
    "Many of you must have tried searching for a friend ",
    "but never found the right one."
]

for el in tag:
  result = TextBlob(el)
  print('\n', el)
  print(result.tags)

# NNP proper noun, singular ‘Harrison’
# CC coordinating conjunction
# VB verbs
# JJ adjective ‘big’


 Sukanya, Rajib and Naba are my good friends. 
[('Sukanya', 'NNP'), ('Rajib', 'NNP'), ('and', 'CC'), ('Naba', 'NNP'), ('are', 'VBP'), ('my', 'PRP$'), ('good', 'JJ'), ('friends', 'NNS')]

 Sukanya is getting married next year. 
[('Sukanya', 'NNP'), ('is', 'VBZ'), ('getting', 'VBG'), ('married', 'VBN'), ('next', 'JJ'), ('year', 'NN')]

 Marriage is a big step in one’s life.
[('Marriage', 'NN'), ('is', 'VBZ'), ('a', 'DT'), ('big', 'JJ'), ('step', 'NN'), ('in', 'IN'), ('one', 'CD'), ('’', 'NN'), ('s', 'JJ'), ('life', 'NN')]

 It is both exciting and frightening. 
[('It', 'PRP'), ('is', 'VBZ'), ('both', 'DT'), ('exciting', 'VBG'), ('and', 'CC'), ('frightening', 'NN')]

 But friendship is a sacred bond between people.
[('But', 'CC'), ('friendship', 'NN'), ('is', 'VBZ'), ('a', 'DT'), ('sacred', 'JJ'), ('bond', 'NN'), ('between', 'IN'), ('people', 'NNS')]

 It is a special kind of love between us. 
[('It', 'PRP'), ('is', 'VBZ'), ('a', 'DT'), ('special', 'JJ'), ('kind', 'NN'), ('of', 'IN'), ('

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


# Chunking

Chunking in Natural Language Processing (NLP) refers to the process of grouping words (or tokens) into chunks, which are meaningful phrases or units based on syntactic structure. This is typically done after part-of-speech (POS) tagging and is often used to identify and extract phrases such as noun phrases (NP), verb phrases (VP), or prepositional phrases (PP).

In [ ]:
input_str = "A black television and a white stove were bought for the new apartment of John."
result = TextBlob(input_str)
print(result.tags)

# DT determiner
# JJ adjective ‘big’
# NN a noun

[('A', 'DT'), ('black', 'JJ'), ('television', 'NN'), ('and', 'CC'), ('a', 'DT'), ('white', 'JJ'), ('stove', 'NN'), ('were', 'VBD'), ('bought', 'VBN'), ('for', 'IN'), ('the', 'DT'), ('new', 'JJ'), ('apartment', 'NN'), ('of', 'IN'), ('John', 'NNP')]


In [ ]:
reg_exp = "NP: {<DT>*<JJ>*<NN>}"
rp = nltk.RegexpParser(reg_exp)
result = rp.parse(result.tags)

for el in result:
  print(el)
# print(result)

(NP A/DT black/JJ television/NN)
('and', 'CC')
(NP a/DT white/JJ stove/NN)
('were', 'VBD')
('bought', 'VBN')
('for', 'IN')
(NP the/DT new/JJ apartment/NN)
('of', 'IN')
('John', 'NNP')


# Entity Recognition

Named Entity Recognition (NER) is a fundamental task in Natural Language Processing (NLP) that involves identifying and classifying named entities in text into predefined categories. Named entities typically include specific items such as people, organizations, locations, dates, numerical values, and more. The goal of NER is to extract useful structured information from unstructured text. Common entity types include ORGANIZATION, PERSON, LOCATION, DATE, TIME, MONEY, and GPE (geo-political entity).

In [ ]:
nltk.download('maxent_ne_chunker')
nltk.download('words')

[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


True

In [ ]:
import nltk
nltk.download('maxent_ne_chunker_tab')

from nltk import word_tokenize, pos_tag, ne_chunk

input_str = "Bill works for Apple so he went to Boston for a conference"
for el in ne_chunk(pos_tag(word_tokenize(input_str))):
  print(el)

[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.


(PERSON Bill/NNP)
('works', 'VBZ')
('for', 'IN')
('Apple', 'NNP')
('so', 'IN')
('he', 'PRP')
('went', 'VBD')
('to', 'TO')
(GPE Boston/NNP)
('for', 'IN')
('a', 'DT')
('conference', 'NN')
